In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from sklearn.covariance import MinCovDet
import os
import time
from sklearn.utils.extmath import fast_logdet

In [2]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd 
import os

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA  # <-- Missing import
import matplotlib.pyplot as plt
#data load
raw_counts = pd.read_csv("gene_counts _for_ml.csv", index_col=0)

In [5]:
raw_counts.columns = raw_counts.columns.str.replace(r'^(CONTROL|TREATMENT).*', r'\1', regex=True)
print("\nCleaned columns:")
print(raw_counts.columns)


Cleaned columns:
Index(['TREATMENT', 'TREATMENT', 'TREATMENT', 'TREATMENT', 'CONTROL',
       'CONTROL', 'CONTROL', 'CONTROL'],
      dtype='object')


In [6]:
filtered_df = raw_counts

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet, LedoitWolf
import hdbscan
from scipy import linalg

def fast_logdet(matrix):
    """Compute log determinant efficiently"""
    try:
        return 2 * np.sum(np.log(np.diag(linalg.cholesky(matrix))))
    except:
        # If Cholesky fails, use SVD-based determinant
        sign, logdet = np.linalg.slogdet(matrix)
        return logdet

# Create output directory
output_dir = "final_results/top3_genes_check_again_distance_comparison"
os.makedirs(output_dir, exist_ok=True)

# 1. Data preparation - Samples as features
normalized_data = np.log2(filtered_df + 1)  # Log2 transform with pseudocount

print(f"Original data shape: {normalized_data.shape}")

# 2. Select top 3000 most variable genes (features)
gene_variability = normalized_data.std(axis=0)  # STD across samples for each gene
top_3k_genes = gene_variability.sort_values(ascending=False).head(3000).index
top_3k_data = normalized_data[top_3k_genes]

print(f"Top 3K variable genes data shape: {top_3k_data.shape}")
print(f"Samples: {top_3k_data.shape[0]}, Features (genes): {top_3k_data.shape[1]}")

# 3. Z-score normalization - scale features (genes) across samples
top_3k_data = pd.DataFrame(
    StandardScaler().fit_transform(top_3k_data),
    index=top_3k_data.index,  # Sample names
    columns=top_3k_data.columns  # Gene names
)

# ----------------------------
class RobustMahalanobisHDBSCAN:
    def __init__(self, min_cluster_size=5, min_samples=None):
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        
    def _robust_mahalanobis(self, X):
        """Calculate pairwise Mahalanobis distances between samples"""
        try:
            n_samples, n_features = X.shape
            
            print(f"Calculating Mahalanobis for {n_samples} samples with {n_features} features")
            
            # For high-dimensional data, use regularized covariance
            if n_features > n_samples:
                print("High-dimensional data: Using Ledoit-Wolf shrinkage")
                cov_estimator = LedoitWolf().fit(X)
            else:
                # Use robust covariance for lower dimensions
                cov_estimator = MinCovDet(support_fraction=0.8).fit(X)
            
            cov = cov_estimator.covariance_
            
            # Regularization to ensure invertibility
            trace_cov = np.trace(cov)
            reg_strength = 1e-6 * trace_cov / n_features
            reg = reg_strength * np.eye(n_features)
            cov_reg = cov + reg
            
            # Check condition number
            cond_number = np.linalg.cond(cov_reg)
            print(f"Covariance matrix condition number: {cond_number:.2e}")
            
            if cond_number > 1e12:
                print("Matrix is ill-conditioned, using pseudo-inverse")
                cov_inv = np.linalg.pinv(cov_reg)
            else:
                cov_inv = np.linalg.inv(cov_reg)
            
            # Calculate Mahalanobis distances between samples
            diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]
            dists = np.sqrt(np.einsum('...i,ij,...j->...', diff, cov_inv, diff))
            np.fill_diagonal(dists, 0)
            
            return dists
            
        except Exception as e:
            print(f"Mahalanobis calculation failed: {str(e)}")
            return None

    def fit(self, X):
        dist_matrix = self._robust_mahalanobis(X)
        if dist_matrix is None:
            print("Using Euclidean as fallback for Mahalanobis")
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='euclidean'
            )
            self.labels_ = self.clusterer.fit_predict(X)
        else:
            self.clusterer = hdbscan.HDBSCAN(
                min_cluster_size=self.min_cluster_size,
                min_samples=self.min_samples,
                metric='precomputed'
            )
            self.labels_ = self.clusterer.fit_predict(dist_matrix)
        return self

# ----------------------------
# Metric Comparison - Clustering SAMPLES based on top 3K variable genes
# ----------------------------
metrics = [
    ('euclidean', hdbscan.HDBSCAN(metric='euclidean')),
    ('manhattan', hdbscan.HDBSCAN(metric='manhattan')),
    ('correlation', hdbscan.HDBSCAN(metric='correlation')),
    ('mahalanobis', RobustMahalanobisHDBSCAN())
]

results = []
for name, clusterer in metrics:
    print(f"\n{'='*40}\nClustering SAMPLES with: {name.upper()}\n{'='*40}")
    
    try:
        start_time = time.time()
        
        if name == 'mahalanobis':
            print("Computing Mahalanobis distances between samples...")
            clusterer.fit(top_3k_data.values)
            clusters = clusterer.labels_
        else:
            clusters = clusterer.fit_predict(top_3k_data.values)
            
        # Analysis
        unique_clusters = np.unique(clusters)
        n_clusters = len(unique_clusters) - 1  # Exclude noise (-1)
        noise_ratio = np.mean(clusters == -1)
        cluster_sizes = [np.sum(clusters == c) for c in unique_clusters if c != -1]
        
        results.append({
            'metric': name,
            'n_clusters': n_clusters,
            'noise_ratio': noise_ratio,
            'clusters': clusters,
            'cluster_sizes': cluster_sizes,
            'time': time.time() - start_time
        })
        
        print(f"Number of sample clusters: {n_clusters}")
        print(f"Noise ratio: {noise_ratio:.1%}")
        print(f"Cluster sizes (samples per cluster): {sorted(cluster_sizes, reverse=True)}")
        print(f"Time elapsed: {results[-1]['time']:.2f} seconds")
        
    except Exception as e:
        print(f"Clustering failed: {str(e)}")
        results.append({
            'metric': name,
            'error': str(e),
            'clusters': None
        })

# ----------------------------
# Visualization - Samples in PCA space
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

print("Computing PCA for sample visualization...")
pca = PCA(n_components=2).fit_transform(top_3k_data.values)

for i, result in enumerate(results):
    ax = axes[i]
    
    if result['clusters'] is None:
        ax.text(0.5, 0.5, f"Failed:\n{result.get('error','')}", 
                ha='center', va='center', fontsize=12)
        ax.set_title(f"{result['metric'].upper()} - FAILED", fontsize=14)
        continue
        
    sc = ax.scatter(
        pca[:, 0], pca[:, 1],
        c=result['clusters'],
        cmap='tab20',
        alpha=0.7,
        s=50
    )
    
    title = (
        f"{result['metric'].upper()}\n"
        f"Sample Clusters: {result['n_clusters']} | "
        f"Noise: {result['noise_ratio']:.1%}\n"
        f"Time: {result['time']:.2f}s"
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    
    plt.colorbar(sc, ax=ax, label='Cluster ID')

plt.tight_layout()
plt.savefig(f"{output_dir}/top3k_sample_clustering_comparison.png", dpi=300, bbox_inches='tight')
plt.close()

# ----------------------------
# Save Results - Sample clustering
# ----------------------------
for result in results:
    if result['clusters'] is not None:
        pd.DataFrame({
            'sample': top_3k_data.index,  # Sample names
            'cluster': result['clusters']
        }).to_csv(f"{output_dir}/top3k_sample_clusters_{result['metric']}.csv", index=False)

# Save summary statistics
summary_data = []
for r in results:
    summary_data.append({
        'metric': r['metric'],
        'n_clusters': r.get('n_clusters', np.nan),
        'noise_ratio': r.get('noise_ratio', np.nan),
        'largest_cluster': max(r['cluster_sizes']) if 'cluster_sizes' in r and r['cluster_sizes'] else np.nan,
        'time_seconds': r.get('time', np.nan),
        'status': 'SUCCESS' if r['clusters'] is not None else 'FAILED'
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(f"{output_dir}/top3k_sample_clustering_summary.csv", index=False)

# Save the list of top 3000 variable genes
pd.DataFrame({'gene': top_3k_genes}).to_csv(f"{output_dir}/top_3000_variable_genes.csv", index=False)

print(f"\nSample clustering analysis complete. Results saved to: {output_dir}")
print(f"Top 3000 variable genes used for clustering")
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))

Original data shape: (28581, 8)
Top 3K variable genes data shape: (28581, 32)
Samples: 28581, Features (genes): 32

Clustering SAMPLES with: EUCLIDEAN


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of sample clusters: 102
Noise ratio: 35.1%
Cluster sizes (samples per cluster): [np.int64(11465), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(29), np.int64(28), np.int64(28), np.int64(25), np.int64(23), np.int64(22), np.int64(21), np.int64(21), np.int64(21), np.int64(19), np.int64(18), np.int64(18), np.int64(18), np.int64(17), np.int64(16), np.int64(16), np.int64(16), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(14), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64

/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of sample clusters: 88
Noise ratio: 17.8%
Cluster sizes (samples per cluster): [np.int64(16594), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(29), np.int64(25), np.int64(25), np.int64(24), np.int64(22), np.int64(21), np.int64(19), np.int64(18), np.int64(18), np.int64(17), np.int64(16), np.int64(16), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(14), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(13), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np

...

/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/covariance/_robust_covariance.py:749: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(


Covariance matrix condition number: 3.19e+07
Mahalanobis calculation failed: Unable to allocate 195. GiB for an array with shape (28581, 28581, 32) and data type float64
Using Euclidean as fallback for Mahalanobis


/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/shafeeq/Downloads/yes/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Number of sample clusters: 102
Noise ratio: 35.1%
Cluster sizes (samples per cluster): [np.int64(11465), np.int64(5400), np.int64(127), np.int64(123), np.int64(92), np.int64(87), np.int64(74), np.int64(71), np.int64(70), np.int64(29), np.int64(28), np.int64(28), np.int64(25), np.int64(23), np.int64(22), np.int64(21), np.int64(21), np.int64(21), np.int64(19), np.int64(18), np.int64(18), np.int64(18), np.int64(17), np.int64(16), np.int64(16), np.int64(16), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(15), np.int64(14), np.int64(14), np.int64(14), np.int64(13), np.int64(13), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64

Exception ignored in: <function ResourceTracker.__del__ at 0x7bc1fe57f920>
Traceback (most recent call last):
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x77f4f2f8f920>
Traceback (most recent call last):
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/shafeeq/Downloads/yes/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in